# FlashNystrom — Colab experiments

Reproduces the paper experiments: **kernel scaling / crossover**, **MQAR recall** (length sweep + capacity ablation), and **CIFAR pixel-token** vision.

**GPU requirement:** the fused kernels are **sm_80+** — use an **A100 or L4** runtime (Colab Pro / Pro+). A **T4 (sm_75) will NOT work.**  
Set it via *Runtime → Change runtime type → GPU → A100* (or L4).

## 0. Check the GPU

In [ ]:
import torch
name = torch.cuda.get_device_name()
cap = torch.cuda.get_device_capability()
print(name, '| compute capability', cap)
assert cap[0] >= 8, (
    f'FlashNystrom kernels need sm_80+ (A100/L4); this GPU is {name} '
    f'(sm_{cap[0]}{cap[1]}). Switch the Colab runtime to A100 or L4.')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Get the code

Push your repo to GitHub first — the `paper/*.py` experiment code is tracked, the manuscript is gitignored — then set `REPO_URL`. If it isn't on GitHub, use one of the commented alternatives (Drive / upload a zip).

In [ ]:
REPO_URL = 'https://github.com/<you>/FlashNystrom.git'   # <-- set this
!git clone $REPO_URL flashnystrom
%cd flashnystrom

# --- alternatives if not on GitHub ---
# from google.colab import drive; drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/FlashNystrom.zip -d /content
# %cd /content/FlashNystrom
#
# from google.colab import files; files.upload()   # pick a repo .zip
# !unzip -q FlashNystrom.zip && %cd FlashNystrom

## 2. Build the fused CUDA kernels

Compiles `csrc/` with nvcc against Colab's preinstalled torch (~3–6 min). We pin the arch to this GPU so the build is fast and correct.

In [ ]:
import os, torch
cap = torch.cuda.get_device_capability()
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'
print('building for sm_' + f'{cap[0]}{cap[1]}')
!pip install -e . --no-build-isolation

## 3. Verify the kernels

Forward must match the pure-PyTorch reference (to bf16 noise) and the backward must be finite.

In [ ]:
import torch
from flash_nystrom import flash_nystrom_attention
from flash_nystrom.reference import nystrom_attention_reference
mk = lambda: torch.randn(4, 2, 256, 64, device='cuda', dtype=torch.bfloat16)
q, k, v = mk(), mk(), mk()
o = flash_nystrom_attention(q, k, v, 64, 6)
r = nystrom_attention_reference(q, k, v, 64, 6)
print('fwd finite:', bool(torch.isfinite(o).all()),
      ' max|fn-ref|:', (o.float() - r.float()).abs().max().item())
qg = q.clone().requires_grad_(True)
flash_nystrom_attention(qg, k, v, 64, 6).sum().backward()
print('bwd grad finite:', bool(torch.isfinite(qg.grad).all()))

## 4. Experiment — kernel scaling / crossover

Training-step throughput + peak memory vs sequence length, at the auto-found max batch (saturates the GPU). Each (backend, N) runs in its own subprocess so one OOM doesn't kill the sweep. This is the figure that shows where flash_nystrom overtakes full attention — and where sdpa OOMs.

In [ ]:
!python benchmarks/profile_scaling.py \
  --backends sdpa flash_nystrom nystrom_reference \
  --Ns 256 512 1024 2048 4096 8192 16384

## 5. Experiment — MQAR recall

**Length sweep:** recall holds as context grows while training cost stays linear (sdpa's blows up / OOMs). **Capacity ablation:** Nyström recall degrades at the rank (landmark) limit — the honest result. Both use auto-batch + `--grad_clip 1.0` and an internal per-config LR sweep (best-over-LR), so they take a while; trim the lists to iterate.

In [ ]:
# Length sweep (fixed kv_pairs, vary seq_len)
!python -m paper.mqar.run_scaling_sweep --mode length \
  --backends sdpa flash_nystrom nystrom_reference \
  --seq_lens 256 512 1024 2048 4096 --num_kv_pairs 16

In [ ]:
# Capacity ablation (fixed seq_len, vary kv_pairs)
!python -m paper.mqar.run_scaling_sweep --mode capacity \
  --backends sdpa flash_nystrom nystrom_reference \
  --seq_len 1024 --kv_pairs 16 32 64 128 256

## 6. Experiment — CIFAR pixel-token (long-context vision)

`patch_size=1` → a 1025-token sequence per image, trained from scratch. Auto-batch + grad clipping. Reports test accuracy, throughput, and peak memory per backend (CIFAR-10 auto-downloads).

In [ ]:
!python benchmarks/train_three_way.py \
  --patch_size 1 --autobatch --epochs 30 --grad_clip 1.0 \
  --backends sdpa nystrom_reference flash_nystrom

## Notes

- Results print to stdout. To keep them: append `> out.txt 2>&1` and download `out.txt`, or save to Drive (`drive.mount` above).
- Run **one experiment per session** to economize GPU time; the MQAR sweeps are the longest.
- Still owed before trusting the numbers: a 3-seed `grad_clip` lock-in and an autobatch-vs-fixed-256 recall check (the batch↔LR↔epochs coupling).